# 5 · Bronze, from somebody else's API

A fourth source, and the first one **we do not control**.

| | |
|---|---|
| **reads** | PayNimbus, a payment processor, over HTTP |
| **writes** | `teach.bronze_settlements` in PostgreSQL |
| **runs** | hourly |

A database answers instantly and always. A partner's API answers **when it feels
like it**, sometimes not at all, and it is entitled to change its mind about
what a field means without telling you.

Everything different about this pipeline follows from that.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import json, os, time, urllib.request
import psycopg
from pipelines.lib.config import dsn, SCHEMA

# Read the address from the environment, never hardcode it. On a laptop this is
# localhost. Inside the Airflow container "localhost" is the container itself,
# and the partner is at http://partner-api:8088. A pipeline that only works in
# one of those two places is not finished.
API = os.environ.get('PARTNER_API', 'http://localhost:8088') + '/v1/settlements/lookup'
print(API)

---

## Step 0 · What are we even asking about?

We hold payments. PayNimbus holds settlements. **They are not the same thing.**

A payment is *"the rider was charged"*. A settlement is *"the money actually
reached our bank"*. Days can pass between the two, and some payments never
settle at all.

In [ ]:
sql("""
    SELECT payment_id, trip_id, amount, status, captured_at
    FROM kerb.payments
    WHERE status = 'captured'
    ORDER BY captured_at DESC
    LIMIT 5
""", 'our side: payments we captured')

## Step 1 · One HTTP call, kept boring

No retries, no backoff, no circuit breaker. Not because those are wrong, but
because they belong in a shared client, and putting them here would bury the
lesson under plumbing.

**The timeout is not optional.** Without it, a partner that hangs takes your
pipeline with it, forever, silently.

In [ ]:
def ask(refs):
    """Ask the partner about a list of our payment ids."""
    request = urllib.request.Request(
        API,
        data=json.dumps({'refs': refs}).encode(),
        headers={'content-type': 'application/json'})
    with urllib.request.urlopen(request, timeout=30) as response:   # <- the timeout
        return json.load(response).get('settlements', [])

with psycopg.connect(dsn()) as c:
    pairs = c.execute("""SELECT payment_id, trip_id FROM kerb.payments
                        WHERE status = 'captured'
                        ORDER BY captured_at DESC LIMIT 10""").fetchall()

refs = [p for p, _ in pairs]
answer = ask(refs)

print(f'we asked about {len(refs)} payments')
print(f'they answered about {len(answer)}\n')
print(json.dumps(answer[0], indent=2))

### Read those two numbers again

We asked about ten. They answered about fewer.

![](img/api-1-outcomes.png)

**The missing ones are not an error.** A charge still in flight is simply not in
the response. Treat absent as a failure and you page somebody every night for a
system behaving exactly as designed.

## See it for yourself

In [ ]:
came_back = {s.get('merchant_ref') for s in answer}

for ref in refs:
    print(f"  {ref}  {'answered' if ref in came_back else 'absent, still in flight'}")

---

## Step 2 · Ask in batches, and pick the size on purpose

We have thousands of payments to ask about. One HTTP call each is thousands of
round trips against a partner's system. That is not a pipeline, that is an
accidental denial of service on a company you have a contract with, and somebody
will phone you about it.

![](img/api-2-batching.png)

In [ ]:
BATCH = 250        # a named constant, so it can be argued about

with psycopg.connect(dsn()) as c:
    pairs = c.execute("""SELECT payment_id, trip_id FROM kerb.payments
                        WHERE status = 'captured'
                        ORDER BY captured_at DESC LIMIT 2000""").fetchall()

trip_of = dict(pairs)                       # the join key we add ourselves, see step 4
refs    = [p for p, _ in pairs]

t0, answers = time.time(), []
for i in range(0, len(refs), BATCH):
    chunk = refs[i:i + BATCH]
    got = ask(chunk)
    answers.extend(got)
    print(f'  asked {len(chunk):>4}, answered {len(got):>4}')

print(f'\n{len(refs):,} payments in {len(refs) // BATCH + 1} calls, {time.time() - t0:.1f}s')

---

## Step 3 · The contract, and why defaulting costs real money

Three answers we know how to file. Look at what the partner actually sends.

In [ ]:
from collections import Counter

KNOWN_STATUS = {'settled', 'failed', 'in_flight'}

seen = Counter(s.get('status') for s in answers)
for status, n_seen in seen.most_common():
    known = 'known' if status in KNOWN_STATUS else 'NOT IN THE CONTRACT'
    print(f'  {str(status)!r:14} {n_seen:>6,}   {known}')

### That empty string is real, and it is in your data right now

Not `failed`. Not `settled`. **`""`**.

An empty status means the money can be neither recognised as revenue nor chased
as missing. It is in a third state that no report has a column for. And look at
what both defaults would cost:

| | |
|---|---|
| default it to `failed` | finance **writes off money that actually arrived** |
| default it to `settled` | finance **books revenue that never came** |

Holding it is the only answer that is not a lie to somebody.

---

## Step 4 · The table, and the column the partner never sends

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.bronze_settlements (
    settlement_id TEXT PRIMARY KEY,   -- the processor's id, so a re-ask is harmless
    payment_id    TEXT,               -- our id, which is what we asked them about
    trip_id       TEXT,               -- added by us. They do not send it
    rail          TEXT,               -- card, upi, wallet
    gross         NUMERIC(12,2),      -- what the rider was charged
    fee           NUMERIC(12,2),      -- what the processor kept
    net           NUMERIC(12,2),      -- what actually reached the bank
    status        TEXT,
    currency      TEXT,
    settled_at    TIMESTAMPTZ
);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

print('table ready')

### Why `trip_id` is in that table

PayNimbus does not send it. **It knows about payments, not rides.**

Without that column this table is a set of numbers with nothing to attach them
to, and a settlement that cannot be joined back to a ride cannot answer any
question anybody actually asks. So we carry it across ourselves, from the
`trip_of` lookup we built in step 2.

---

## Step 5 · Sort every answer, and count all three outcomes

In [ ]:
INSERT = f"""
    INSERT INTO {SCHEMA}.bronze_settlements
        (settlement_id, payment_id, trip_id, rail, gross, fee, net, status, currency, settled_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (settlement_id) DO NOTHING
"""

rows, held = [], []
for s in answers:
    status = s.get('status')
    if status not in KNOWN_STATUS:
        held.append((s.get('merchant_ref'),
                     f'status {status!r} is not one of {sorted(KNOWN_STATUS)}'))
        continue

    rows.append((
        s['settlement_id'],
        s.get('merchant_ref'),                    # their name for our payment_id
        trip_of.get(s.get('merchant_ref')),       # the join key we added
        s.get('rail'), s.get('gross'), s.get('fee'), s.get('net'),
        status, s.get('currency'), s.get('settled_at')))

with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    cur.executemany(INSERT, rows)
    c.commit()

absent = len(refs) - len(rows) - len(held)
print(f'asked   {len(refs):>6,}')
print(f'written {len(rows):>6,}   they answered and we understood')
print(f'held    {len(held):>6,}   they answered and we did not')
print(f'absent  {absent:>6,}   they did not answer. Still in flight')

**Three numbers that always add up to what you asked for.** If they ever do not,
you have a bug, and you can see it in one line.

---

## And now the packaged pipeline

In [ ]:
run('-m', 'pipelines.p4_bronze_settlements', '--limit', '5000')

In [ ]:
sql(f"""
    SELECT pipeline, status, rows_in, rows_out,
           round(extract(epoch from (ended_at - started_at))::numeric, 2) AS secs, message
    FROM {SCHEMA}.runs
    WHERE pipeline = 'p4_bronze_settlements'
    ORDER BY started_at DESC LIMIT 3
""", 'the run log')

In [ ]:
sql(f"""
    SELECT status, rail, count(*) AS settlements,
           to_char(sum(gross), '999,999,999.99') AS gross,
           to_char(sum(fee),   '999,999,999.99') AS fee
    FROM {SCHEMA}.bronze_settlements
    GROUP BY 1, 2 ORDER BY 1, 3 DESC
""", 'what landed')

## And what is being held

In [ ]:
sql(f"""
    SELECT reason, count(*) AS records
    FROM {SCHEMA}.quarantine
    WHERE pipeline = 'p4_bronze_settlements'
    GROUP BY 1 ORDER BY 2 DESC
""", 'held, with the reason attached')

---

## What you learned

- A partner's API is **not a database**. It answers late, partially, or not at all
- **Three outcomes, not two**: written, held, absent
- **Absent is healthy.** Money that has not moved has nothing to report
- Batch your requests, and put the batch size in **a named constant**
- **Always set a timeout.** A hanging partner should not hang you
- An unreadable status is held. Defaulting it either writes off real money or
  books revenue that never came
- If the partner cannot give you the join key, **carry it across yourself**